<a href="https://colab.research.google.com/github/vishnuvadhan7/Eco-tourism-Climate-Risk-Prediction/blob/main/Eco_tourism_Climate_Risk_Prediction_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%pip install lime
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
import plotly.figure_factory as ff
from plotly.subplots import make_subplots
import folium
from folium import plugins

# Machine Learning Libraries
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder, MinMaxScaler
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier, IsolationForest
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.svm import SVR, SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.cluster import KMeans, DBSCAN
from sklearn.metrics import (mean_squared_error, r2_score, accuracy_score,
                           classification_report, confusion_matrix,
                           silhouette_score, davies_bouldin_score)
from sklearn.pipeline import Pipeline

# Advanced ML Libraries
import xgboost as xgb
import lightgbm as lgb

# Explainable AI
import shap
import lime
from lime.lime_tabular import LimeTabularExplainer

# Suppress warnings
import warnings
warnings.filterwarnings('ignore')

# Set random seed for reproducibility
np.random.seed(42)

# Configure plotting
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (12, 8)

In [ ]:
def load_ecotourism_data(filepath='/content/ecotourism_dataset.csv'):
    try:
        df = pd.read_csv(filepath)
        if df.empty:
            print(f"Warning: The dataset at {filepath} is empty.")
            return None
        print(f"Successfully loaded data from {filepath}. Shape: {df.shape}")
        return df
    except FileNotFoundError:
        print(f"Error: File not found at {filepath}")
        return None
    except Exception as e:
        print(f"An error occurred: {e}")
        return None

# Attempt to load the data
df = load_ecotourism_data()
df.head(5)

Successfully loaded data from /content/ecotourism_dataset.csv. Shape: (5000, 16)


,Location,Latitude,Longitude,Vegetation_Type,Biodiversity_Index,Protected_Area,Elevation_m,Slope_Degree,Soil_Type,Air_Quality_Index,Average_Temperature_C,Flood_Risk,Tourist_Attractions,Accessibility_Score,Tourist_Capacity_Limit,Climate_Risk_Score
0,Site 1,24.981605,-102.445514,Wetland,0.903357,False,2901,28,Sandy,129,9.772177,Moderate,6,8,958,0.626427
1,Site 2,48.028572,-96.859504,Mountain,0.508270,True,257,34,Clay,92,-8.684627,Moderate,4,4,485,0.275427
2,Site 3,39.279758,-70.181682,Mountain,0.593404,False,3459,48,Sandy,104,0.721349,High,3,9,112,0.592100
3,Site 4,33.946339,-106.199693,Grassland,0.827453,False,44,36,Clay,124,13.825356,High,4,4,602,0.681884
4,Site 5,16.240746,-69.124522,Forest,0.687880,False,268,20,Clay,72,2.958583,High,8,2,792,0.396066


In [ ]:
n_samples = 5000
data = {
        'Site_ID': range(1, n_samples + 1),
        'Site_Name': [f'EcoSite_{i}' for i in range(1, n_samples + 1)],
        'Latitude': np.random.uniform(10, 50, n_samples),
        'Longitude': np.random.uniform(-130, -60, n_samples),
        'Country': np.random.choice(['USA', 'Canada', 'Mexico', 'Costa Rica', 'Brazil'], n_samples),
        'Biodiversity_Index': np.random.uniform(0.2, 1.0, n_samples),
        'Air_Quality_Index': np.random.uniform(20, 200, n_samples),
        'Elevation_m': np.random.uniform(0, 4000, n_samples),
        'Vegetation_Type': np.random.choice(['Tropical_Forest', 'Temperate_Forest', 'Grassland',
                                           'Desert', 'Wetland', 'Mountain'], n_samples),
        'Flood_Risk_Index': np.random.uniform(0, 1, n_samples),
        'Drought_Risk_Index': np.random.uniform(0, 1, n_samples),
        'Temperature_C': np.random.uniform(5, 40, n_samples),
        'Annual_Rainfall_mm': np.random.uniform(200, 3000, n_samples),
        'Soil_Type': np.random.choice(['Clay', 'Sand', 'Loam', 'Rocky', 'Volcanic'], n_samples),
        'Soil_Erosion_Risk': np.random.uniform(0, 1, n_samples),
        'Tourist_Capacity_Limit': np.random.uniform(100, 5000, n_samples),
        'Current_Tourist_Count': np.random.uniform(50, 4000, n_samples),
        'Accessibility_Score': np.random.uniform(0.1, 1.0, n_samples),
        'Human_Activity_Index': np.random.uniform(0.05, 0.95, n_samples),
        'Protected_Area_Status': np.random.choice([True, False], n_samples),
        'Conservation_Investment_USD': np.random.uniform(10000, 500000, n_samples),
        'Climate_Risk_Score': np.random.uniform(0.1, 0.9, n_samples)
    }
df = pd.DataFrame(data)
df.to_csv('eco_sites.csv', index=False)

In [ ]:
n_samples = 5000
data = {
        'Site_ID': range(1, n_samples + 1),
        'Site_Name': [f'EcoSite_{i}' for i in range(1, n_samples + 1)],
        'Latitude': np.random.uniform(10, 50, n_samples),
        'Longitude': np.random.uniform(-130, -60, n_samples),
        'Country': np.random.choice(['USA', 'Canada', 'Mexico', 'Costa Rica', 'Brazil'], n_samples),
        'Biodiversity_Index': np.random.uniform(0.2, 1.0, n_samples),
        'Air_Quality_Index': np.random.uniform(20, 200, n_samples),
        'Elevation_m': np.random.uniform(0, 4000, n_samples),
        'Vegetation_Type': np.random.choice(['Tropical_Forest', 'Temperate_Forest', 'Grassland',
                                           'Desert', 'Wetland', 'Mountain'], n_samples),
        'Flood_Risk_Index': np.random.uniform(0, 1, n_samples),
        'Drought_Risk_Index': np.random.uniform(0, 1, n_samples),
        'Temperature_C': np.random.uniform(5, 40, n_samples),
        'Annual_Rainfall_mm': np.random.uniform(200, 3000, n_samples),
        'Soil_Type': np.random.choice(['Clay', 'Sand', 'Loam', 'Rocky', 'Volcanic'], n_samples),
        'Soil_Erosion_Risk': np.random.uniform(0, 1, n_samples),
        'Tourist_Capacity_Limit': np.random.uniform(100, 5000, n_samples),
        'Current_Tourist_Count': np.random.uniform(50, 4000, n_samples),
        'Accessibility_Score': np.random.uniform(0.1, 1.0, n_samples),
        'Human_Activity_Index': np.random.uniform(0.05, 0.95, n_samples),
        'Protected_Area_Status': np.random.choice([True, False], n_samples),
        'Conservation_Investment_USD': np.random.uniform(10000, 500000, n_samples),
        'Climate_Risk_Score': np.random.uniform(0.1, 0.9, n_samples)
    }
df = pd.DataFrame(data)
df['Vulnerability_Score'] = (
        0.25 * (1 - df['Biodiversity_Index']) +
        0.15 * (df['Air_Quality_Index'] / 200) +
        0.15 * df['Flood_Risk_Index'] +
        0.15 * df['Drought_Risk_Index'] +
        0.10 * df['Soil_Erosion_Risk'] +
        0.10 * (df['Temperature_C'] / 40) +
        0.10 * df['Climate_Risk_Score']
    )
df['Vulnerability_Score'] += np.random.normal(0, 0.05, n_samples)
df['Vulnerability_Score'] = np.clip(df['Vulnerability_Score'], 0, 1)
df['Risk_Category'] = pd.cut(
        df['Vulnerability_Score'],
        bins=[0, 0.33, 0.67, 1.0],
        labels=['Low', 'Medium', 'High']
    )
df.head()

,Site_ID,Site_Name,Latitude,Longitude,Country,Biodiversity_Index,Air_Quality_Index,Elevation_m,Vegetation_Type,Flood_Risk_Index,...,Soil_Erosion_Risk,Tourist_Capacity_Limit,Current_Tourist_Count,Accessibility_Score,Human_Activity_Index,Protected_Area_Status,Conservation_Investment_USD,Climate_Risk_Score,Vulnerability_Score,Risk_Category
0,1,EcoSite_1,12.128953,-112.831153,USA,0.362570,27.793474,751.379469,Desert,0.343144,...,0.688007,1132.762585,1954.733867,0.776266,0.634815,True,225337.356127,0.803116,0.542223,Medium
1,2,EcoSite_2,38.712710,-119.484041,Brazil,0.891428,195.366761,808.855331,Grassland,0.045769,...,0.877113,413.880509,2583.621547,0.151106,0.661583,True,421170.039580,0.867493,0.478458,Medium
2,3,EcoSite_3,24.272005,-129.557408,Brazil,0.343883,127.992074,152.501865,Desert,0.267393,...,0.770371,1636.150070,3008.028798,0.944625,0.425925,True,380063.405031,0.874163,0.610021,Medium
3,4,EcoSite_4,42.720031,-63.829039,Costa Rica,0.367829,99.074512,374.768245,Tropical_Forest,0.305476,...,0.638754,3550.114106,2266.720286,0.516182,0.574249,False,365670.852963,0.716814,0.507130,Medium
4,5,EcoSite_5,25.507561,-75.105399,Brazil,0.698273,151.864063,1695.498390,Temperate_Forest,0.390106,...,0.316348,3224.721403,3587.848933,0.447712,0.219968,True,194178.920222,0.171431,0.428003,Medium


In [ ]:
def create_comprehensive_eda(df):
 fig = plt.figure(figsize=(24, 20))

 # 1. Distribution of Vulnerability Score
 plt.subplot(4, 4, 1)
 plt.hist(df['Vulnerability_Score'], bins=40, alpha=0.7, color='skyblue', edgecolor='black')
 plt.title('Distribution of Vulnerability Scores', fontsize=12, fontweight='bold')
 plt.xlabel('Vulnerability Score')
 plt.ylabel('Frequency')

 # 2. Risk Category Distribution
 plt.subplot(4, 4, 2)
 risk_counts = df['Risk_Category'].value_counts()
 plt.pie(risk_counts.values, labels=risk_counts.index, autopct='%1.1f%%', startangle=90)
 plt.title('Risk Category Distribution', fontsize=12, fontweight='bold')

 # 3. Biodiversity vs Vulnerability
 plt.subplot(4, 4, 3)
 plt.scatter(df['Biodiversity_Index'], df['Vulnerability_Score'], alpha=0.6, c='green', s=2)
 plt.title('Biodiversity vs Vulnerability', fontsize=12, fontweight='bold')
 plt.xlabel('Biodiversity Index')
 plt.ylabel('Vulnerability Score')

 # 4. Air Quality Impact
 plt.subplot(4, 4, 4)
 plt.scatter(df['Air_Quality_Index'], df['Vulnerability_Score'], alpha=0.6, c='orange', s=2)
 plt.title('Air Quality vs Vulnerability', fontsize=12, fontweight='bold')
 plt.xlabel('Air Quality Index')
 plt.ylabel('Vulnerability Score')

 # 5. Temperature vs Vulnerability
 plt.subplot(4, 4, 5)
 plt.scatter(df['Temperature_C'], df['Vulnerability_Score'], alpha=0.6, c='red', s=2)
 plt.title('Temperature vs Vulnerability', fontsize=12, fontweight='bold')
 plt.xlabel('Temperature (°C)')
 plt.ylabel('Vulnerability Score')

 # 6. Elevation Distribution by Risk Category
 plt.subplot(4, 4, 6)
 for risk_cat in df['Risk_Category'].unique():
  if pd.notna(risk_cat):
   subset = df[df['Risk_Category'] == risk_cat]
   plt.hist(subset['Elevation_m'], alpha=0.6, label=risk_cat, bins=20)
 plt.title('Elevation by Risk Category', fontsize=12, fontweight='bold')
 plt.xlabel('Elevation (m)')
 plt.ylabel('Frequency')
 plt.legend()

 # 7. Flood Risk vs Drought Risk
 plt.subplot(4, 4, 7)
 scatter = plt.scatter(df['Flood_Risk_Index'], df['Drought_Risk_Index'],
 c=df['Vulnerability_Score'], cmap='Reds', alpha=0.6, s=3)
 plt.title('Flood vs Drought Risk', fontsize=12, fontweight='bold')
 plt.xlabel('Flood Risk Index')
 plt.ylabel('Drought Risk Index')
 plt.colorbar(scatter, shrink=0.8)

 # 8. Vegetation Type Analysis
 plt.subplot(4, 4, 8)
 veg_vuln = df.groupby('Vegetation_Type')['Vulnerability_Score'].mean().sort_values()
 plt.barh(veg_vuln.index, veg_vuln.values, color='lightgreen', edgecolor='black')
 plt.title('Avg Vulnerability by Vegetation', fontsize=12, fontweight='bold')
 plt.xlabel('Average Vulnerability Score')

 # 9. Geographic Distribution
 plt.subplot(4, 4, 9)
 scatter = plt.scatter(df['Longitude'], df['Latitude'],
 c=df['Vulnerability_Score'], cmap='RdYlBu_r', alpha=0.6, s=1)
 plt.title('Geographic Vulnerability Distribution', fontsize=12, fontweight='bold')
 plt.xlabel('Longitude')
 plt.ylabel('Latitude')
 plt.colorbar(scatter, shrink=0.8)

 # 10. Tourist Capacity vs Current Count
 plt.subplot(4, 4, 10)
 plt.scatter(df['Tourist_Capacity_Limit'], df['Current_Tourist_Count'],
 c=df['Vulnerability_Score'], cmap='viridis', alpha=0.6, s=2)
 plt.title('Tourist Capacity vs Current Count', fontsize=12, fontweight='bold')
 plt.xlabel('Capacity Limit')
 plt.ylabel('Current Count')

 # 11. Conservation Investment Impact
 plt.subplot(4, 4, 11)
 plt.scatter(df['Conservation_Investment_USD'], df['Vulnerability_Score'],
 alpha=0.6, c='purple', s=2)
 plt.title('Conservation Investment vs Risk', fontsize=12, fontweight='bold')
 plt.xlabel('Investment (USD)')
 plt.ylabel('Vulnerability Score')

 # 12. Protected Area Status
 plt.subplot(4, 4, 12)
 protected_vuln = df.groupby('Protected_Area_Status')['Vulnerability_Score'].mean()
 plt.bar(['Not Protected', 'Protected'], protected_vuln.values,
 color=['red', 'green'], alpha=0.7, edgecolor='black')
 plt.title('Protection Status vs Vulnerability', fontsize=12, fontweight='bold')
 plt.ylabel('Average Vulnerability Score')

 # 13. Correlation Heatmap
 plt.subplot(4, 4, 13)
 numeric_cols = df.select_dtypes(include=[np.number]).columns
 corr_matrix = df[numeric_cols].corr()

 # Select key correlations with vulnerability
 key_corr = corr_matrix['Vulnerability_Score'].sort_values(key=abs, ascending=False)[:10]
 plt.barh(range(len(key_corr)), key_corr.values, color='steelblue')
 plt.yticks(range(len(key_corr)), key_corr.index, fontsize=10)
 plt.title('Top Correlations with Vulnerability', fontsize=12, fontweight='bold')
 plt.xlabel('Correlation Coefficient')

 # 14. Soil Type Distribution
 plt.subplot(4, 4, 14)
 soil_counts = df['Soil_Type'].value_counts()
 plt.pie(soil_counts.values, labels=soil_counts.index, autopct='%1.1f%%')
 plt.title('Soil Type Distribution', fontsize=12, fontweight='bold')

 # 15. Human Activity vs Vulnerability
 plt.subplot(4, 4, 15)
 plt.scatter(df['Human_Activity_Index'], df['Vulnerability_Score'],
 alpha=0.6, c='brown', s=2)
 plt.title('Human Activity vs Vulnerability', fontsize=12, fontweight='bold')
 plt.xlabel('Human Activity Index')
 plt.ylabel('Vulnerability Score')

 # 16. Accessibility Score Distribution
 plt.subplot(4, 4, 16)
 plt.hist(df['Accessibility_Score'], bins=30, alpha=0.7, color='gold', edgecolor='black')
 plt.title('Accessibility Score Distribution', fontsize=12, fontweight='bold')
 plt.xlabel('Accessibility Score')
 plt.ylabel('Frequency')

 plt.tight_layout()
 plt.savefig('comprehensive_eda_analysis.png', dpi=300, bbox_inches='tight')
 plt.show()
 create_comprehensive_eda(df)

In [ ]:
from sklearn.preprocessing import LabelEncoder

def preprocess_data(df, target_column='Vulnerability_Score', task_type='regression'):
    print("\n" + "="*50)
    print(f"DATA PREPROCESSING - {task_type.upper()}")
    print("="*50)
    processed_df = df.copy()

    # Handle missing values
    if processed_df.isnull().sum().sum() > 0:
        numeric_columns = processed_df.select_dtypes(include=[np.number]).columns
        categorical_columns = processed_df.select_dtypes(include=['object', 'category']).columns
        # Fill numeric columns with median
        for col in numeric_columns:
            if processed_df[col].isnull().sum() > 0:
                processed_df[col].fillna(processed_df[col].median(), inplace=True)
        # Fill categorical columns with mode
        for col in categorical_columns:
            if processed_df[col].isnull().sum() > 0:
                processed_df[col].fillna(processed_df[col].mode()[0], inplace=True)

    # Prepare feature matrix X
    X = processed_df.copy()

    # Encode categorical variables
    categorical_columns_to_encode = ['Vegetation_Type', 'Soil_Type', 'Country']
    label_encoders = {}
    for col in categorical_columns_to_encode:
        if col in X.columns:
            le = LabelEncoder()
            X[col] = le.fit_transform(X[col].astype(str))
            label_encoders[col] = le
            print(f" Encoded {col}: {len(le.classes_)} categories")

    # Handle boolean columns
    if 'Protected_Area_Status' in X.columns:
        X['Protected_Area_Status'] = X['Protected_Area_Status'].astype(int)

    # Remove non-feature columns
    columns_to_remove = ['Site_ID', 'Site_Name', target_column]
    if task_type == 'classification' and 'Risk_Category' in X.columns:
        columns_to_remove.append('Risk_Category')
    elif task_type == 'regression' and 'Risk_Category' in X.columns: # Exclude Risk_Category from features for regression
         columns_to_remove.append('Risk_Category')


    for col in columns_to_remove:
        if col in X.columns:
            X = X.drop(col, axis=1)

    # Prepare target variable y
    if task_type == 'regression':
        y = processed_df[target_column]
        print(f" Target variable: {target_column} (continuous)")
    else:  # classification
        if 'Risk_Category' in processed_df.columns:
            le_target = LabelEncoder()
            y = le_target.fit_transform(processed_df['Risk_Category'])
            print(f" Target variable: Risk_Category (categorical)")
            print(f"   Classes: {le_target.classes_}")
        else:
            # Create categories from continuous target
            y = pd.cut(processed_df[target_column], bins=3, labels=[0, 1, 2])
            print(f" Target variable: Created from {target_column} (3 categories)")

    print(f"\n📊 Preprocessing Summary:")
    print(f"   Features: {X.shape[1]}")
    print(f"   Samples: {X.shape[0]}")


    return X, y, label_encoders

# Preprocess data for both regression and classification
X_reg, y_reg, encoders_reg = preprocess_data(df, task_type='regression')
X_clf, y_clf, encoders_clf = preprocess_data(df, task_type='classification')


DATA PREPROCESSING - REGRESSION
 Encoded Vegetation_Type: 6 categories
 Encoded Soil_Type: 5 categories
 Encoded Country: 5 categories
 Target variable: Vulnerability_Score (continuous)

📊 Preprocessing Summary:
   Features: 20
   Samples: 5000

DATA PREPROCESSING - CLASSIFICATION
 Encoded Vegetation_Type: 6 categories
 Encoded Soil_Type: 5 categories
 Encoded Country: 5 categories
 Target variable: Risk_Category (categorical)
   Classes: ['High' 'Low' 'Medium']

📊 Preprocessing Summary:
   Features: 20
   Samples: 5000


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

def split_and_scale_data(X, y, test_size=0.2, task_type='regression'):
    print("\n" + "="*50)
    print(f"DATA SPLITTING AND SCALING - {task_type.upper()}")
    print("="*50)

    # Train-test split
    if task_type == 'classification':
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=test_size, random_state=42, stratify=y
        )
    else:
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=test_size, random_state=42
        )

    # Feature scaling
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    print(f" Data split completed:")
    print(f"   Training set: {X_train.shape}")
    print(f"   Test set: {X_test.shape}")
    print(f"   Feature scaling: StandardScaler applied")

    return X_train, X_test, y_train, y_test, X_train_scaled, X_test_scaled, scaler

# Split and scale data for both tasks
X_train_reg, X_test_reg, y_train_reg, y_test_reg, X_train_reg_scaled, X_test_reg_scaled, scaler_reg = split_and_scale_data(X_reg, y_reg, task_type='regression')
X_train_clf, X_test_clf, y_train_clf, y_test_clf, X_train_clf_scaled, X_test_clf_scaled, scaler_clf = split_and_scale_data(X_clf, y_clf, task_type='classification')


DATA SPLITTING AND SCALING - REGRESSION
 Data split completed:
   Training set: (4000, 20)
   Test set: (1000, 20)
   Feature scaling: StandardScaler applied

DATA SPLITTING AND SCALING - CLASSIFICATION
 Data split completed:
   Training set: (4000, 20)
   Test set: (1000, 20)
   Feature scaling: StandardScaler applied


In [ ]:
from sklearn.model_selection import GridSearchCV, cross_val_score
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
import xgboost as xgb


def train_regression_models(X_train, X_test, y_train, y_test):
   # Define models with hyperparameter grids
    models = {
        'Linear Regression': {
            'model': Pipeline([
                ('scaler', StandardScaler()),
                ('regressor', LinearRegression())
            ]),
            'params': {}
        },

        'XGBoost': {
            'model': Pipeline([
                ('scaler', StandardScaler()),
                ('regressor', xgb.XGBRegressor(random_state=42, eval_metric='rmse'))
            ]),
            'params': {
                'regressor__n_estimators': [100, 200],
                'regressor__max_depth': [3, 6],
                'regressor__learning_rate': [0.1, 0.01]
            }
        },

        'Support Vector Regression': {
            'model': Pipeline([
                ('scaler', StandardScaler()),
                ('regressor', SVR())
            ]),
            'params': {
                'regressor__C': [1, 10],
                'regressor__kernel': ['rbf', 'linear']
            }
        }
    }

    results = {}
    best_models = {}
    for name, model_config in models.items():
        print(f" Training {name}...")

        try:
            if model_config['params']:
                # Grid search for hyperparameter tuning
                grid_search = GridSearchCV(
                    model_config['model'],
                    model_config['params'],
                    cv=5,
                    scoring='r2',
                    n_jobs=-1
                )
                grid_search.fit(X_train, y_train)
                best_model = grid_search.best_estimator_
                print(f"   Best params: {grid_search.best_params_}")
            else:
                best_model = model_config['model']
                best_model.fit(X_train, y_train)

            # Make predictions
            train_pred = best_model.predict(X_train)
            test_pred = best_model.predict(X_test)

            # Calculate metrics
            train_r2 = r2_score(y_train, train_pred)
            test_r2 = r2_score(y_test, test_pred)
            rmse = np.sqrt(mean_squared_error(y_test, test_pred))
            mae = np.mean(np.abs(y_test - test_pred))

            # Cross-validation score
            cv_scores = cross_val_score(best_model, X_train, y_train, cv=5, scoring='r2')

            results[name] = {
                'train_r2': train_r2,
                'test_r2': test_r2,
                'rmse': rmse,
                'mae': mae,
                'cv_mean': cv_scores.mean(),
                'cv_std': cv_scores.std(),
                'predictions': test_pred
            }

            best_models[name] = best_model

            print(f" Train R²: {train_r2:.4f}")
            print(f" Test R²: {test_r2:.4f}")
            print(f" RMSE: {rmse:.4f}")
            print(f" MAE: {mae:.4f}")
            print(f" CV Score: {cv_scores.mean():.4f} (+/- {cv_scores.std()*2:.4f})")

        except Exception as e:
            print(f" Error training {name}: {str(e)}")
            continue

    return results, best_models
# Train regression models
regression_results, regression_models = train_regression_models(X_train_reg, X_test_reg, y_train_reg, y_test_reg)

 Training Linear Regression...
 Train R²: 0.8048
 Test R²: 0.8071
 RMSE: 0.0510
 MAE: 0.0406
 CV Score: 0.8020 (+/- 0.0061)
 Training XGBoost...
   Best params: {'regressor__learning_rate': 0.1, 'regressor__max_depth': 3, 'regressor__n_estimators': 100}
 Train R²: 0.8418
 Test R²: 0.7796
 RMSE: 0.0545
 MAE: 0.0432
 CV Score: 0.7767 (+/- 0.0108)
 Training Support Vector Regression...
   Best params: {'regressor__C': 1, 'regressor__kernel': 'linear'}
 Train R²: 0.8015
 Test R²: 0.8058
 RMSE: 0.0511
 MAE: 0.0408
 CV Score: 0.7992 (+/- 0.0075)


In [ ]:
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, cross_val_score # Import RandomizedSearchCV
from sklearn.metrics import accuracy_score, classification_report
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
import xgboost as xgb


def train_classification_models(X_train, y_train, X_test, y_test):
    """Train multiple classification models with hyperparameter tuning"""
    print("\n" + "="*60)
    print("CLASSIFICATION MODELS TRAINING")
    print("="*60)

    # Define models with hyperparameter grids
    models = {
        'Logistic Regression': {
            'model': Pipeline([
                ('scaler', StandardScaler()),
                ('classifier', LogisticRegression(random_state=42, max_iter=1000))
            ]),
            'params': {
                'classifier__C': [0.1, 1, 10],
                'classifier__solver': ['liblinear', 'lbfgs']
            }
        },

        'Random Forest': {
            'model': Pipeline([
                ('scaler', StandardScaler()),
                ('classifier', RandomForestClassifier(random_state=42))
            ]),
            'params': {
                'classifier__n_estimators': [100, 200],
                'classifier__max_depth': [10, 20, None],
                'classifier__min_samples_split': [2, 5]
            }
        },

        'XGBoost': {
            'model': Pipeline([
                ('scaler', StandardScaler()),
                ('classifier', xgb.XGBClassifier(random_state=42, use_label_encoder=False, eval_metric='mlogloss')) # Add use_label_encoder=False to suppress warning
            ]),
            'params': {
                'classifier__n_estimators': [100, 200, 300],
                'classifier__max_depth': [3, 5, 7],
                'classifier__learning_rate': [0.1, 0.05, 0.01],
                'classifier__subsample': [0.7, 0.9, 1.0],
                'classifier__colsample_bytree': [0.7, 0.9, 1.0]
            },
             'tuning_method': 'random' # Indicate using RandomizedSearchCV
        },

        'Gaussian Naive Bayes': {
            'model': Pipeline([
                ('scaler', StandardScaler()),
                ('classifier', GaussianNB())
            ]),
            'params': {
                'classifier__var_smoothing': [1e-9, 1e-8, 1e-7]
            }
        }
    }

    results = {}
    best_models = {}

    for name, model_config in models.items():
        print(f"\n Training {name}...")

        try:
            if model_config['params']:
                if model_config.get('tuning_method') == 'random':
                     # Randomized search for hyperparameter tuning
                     random_search = RandomizedSearchCV(
                         model_config['model'],
                         model_config['params'],
                         n_iter=20, # Number of parameter settings that are sampled.
                         cv=5,
                         scoring='accuracy',
                         n_jobs=-1,
                         random_state=42
                     )
                     random_search.fit(X_train, y_train)
                     best_model = random_search.best_estimator_
                     print(f"   Best params (Randomized Search): {random_search.best_params_}")
                else:
                    # Grid search for hyperparameter tuning (default)
                    grid_search = GridSearchCV(
                        model_config['model'],
                        model_config['params'],
                        cv=5,
                        scoring='accuracy',
                        n_jobs=-1
                    )
                    grid_search.fit(X_train, y_train)
                    best_model = grid_search.best_estimator_
                    print(f"   Best params (Grid Search): {grid_search.best_params_}")
            else:
                best_model = model_config['model']
                best_model.fit(X_train, y_train)

            # Make predictions
            train_pred = best_model.predict(X_train)
            test_pred = best_model.predict(X_test)

            # Calculate metrics
            train_acc = accuracy_score(y_train, train_pred)
            test_acc = accuracy_score(y_test, test_pred)

            # Cross-validation score
            cv_scores = cross_val_score(best_model, X_train, y_train, cv=5, scoring='accuracy')

            results[name] = {
                'train_accuracy': train_acc,
                'test_accuracy': test_acc,
                'cv_mean': cv_scores.mean(),
                'cv_std': cv_scores.std(),
                'predictions': test_pred,
                'classification_report': classification_report(y_test, test_pred, output_dict=True)
            }

            best_models[name] = best_model

            print(f"   Train Accuracy: {train_acc:.4f}")
            print(f"   Test Accuracy: {test_acc:.4f}")
            print(f"   CV Score: {cv_scores.mean():.4f} (+/- {cv_scores.std()*2:.4f})")

            # Print classification report
            print(f"\n   Classification Report for {name}:")
            print(classification_report(y_test, test_pred, target_names=['Low', 'Medium', 'High']))

        except Exception as e:
            print(f"    Error training {name}: {str(e)}")
            continue

    return results, best_models

# Train classification models
classification_results, classification_models = train_classification_models(X_train_clf, y_train_clf, X_test_clf, y_test_clf)


CLASSIFICATION MODELS TRAINING

 Training Logistic Regression...
   Best params (Grid Search): {'classifier__C': 1, 'classifier__solver': 'lbfgs'}
   Train Accuracy: 0.9107
   Test Accuracy: 0.9110
   CV Score: 0.9027 (+/- 0.0168)

   Classification Report for Logistic Regression:
              precision    recall  f1-score   support

         Low       0.81      0.45      0.58        55
      Medium       0.75      0.59      0.66        87
        High       0.93      0.97      0.95       858

    accuracy                           0.91      1000
   macro avg       0.83      0.67      0.73      1000
weighted avg       0.90      0.91      0.90      1000


 Training Random Forest...
   Best params (Grid Search): {'classifier__max_depth': 20, 'classifier__min_samples_split': 5, 'classifier__n_estimators': 100}
   Train Accuracy: 0.9948
   Test Accuracy: 0.8870
   CV Score: 0.8747 (+/- 0.0024)

   Classification Report for Random Forest:
              precision    recall  f1-score   supp

In [ ]:
def compare_model_performance(regression_results, classification_results):
    """Create comprehensive model comparison"""
    print("\n" + "="*60)
    print("MODEL PERFORMANCE COMPARISON")
    print("="*60)

    # Regression Models Comparison
    print("\n📊 REGRESSION MODELS COMPARISON:")
    print("-" * 40)
    reg_comparison = pd.DataFrame({
        'Model': list(regression_results.keys()),
        'Train_R2': [regression_results[m]['train_r2'] for m in regression_results.keys()],
        'Test_R2': [regression_results[m]['test_r2'] for m in regression_results.keys()],
        'RMSE': [regression_results[m]['rmse'] for m in regression_results.keys()],
        'MAE': [regression_results[m]['mae'] for m in regression_results.keys()],
        'CV_Mean': [regression_results[m]['cv_mean'] for m in regression_results.keys()]
    }).sort_values('Test_R2', ascending=False)

    print(reg_comparison.to_string(index=False, float_format='%.4f'))

    # Classification Models Comparison
    print("\n📊 CLASSIFICATION MODELS COMPARISON:")
    print("-" * 40)
    clf_comparison = pd.DataFrame({
        'Model': list(classification_results.keys()),
        'Train_Accuracy': [classification_results[m]['train_accuracy'] for m in classification_results.keys()],
        'Test_Accuracy': [classification_results[m]['test_accuracy'] for m in classification_results.keys()],
        'CV_Mean': [classification_results[m]['cv_mean'] for m in classification_results.keys()]
    }).sort_values('Test_Accuracy', ascending=False)

    print(clf_comparison.to_string(index=False, float_format='%.4f'))

    # Best models
    best_reg_model = reg_comparison.iloc[0]['Model']
    best_clf_model = clf_comparison.iloc[0]['Model']

    print(f"\n Best Regression Model: {best_reg_model}")
    print(f" Best Classification Model: {best_clf_model}")

    return reg_comparison, clf_comparison, best_reg_model, best_clf_model

# Compare model performance
reg_comparison, clf_comparison, best_reg_model, best_clf_model = compare_model_performance(regression_results, classification_results)



MODEL PERFORMANCE COMPARISON

📊 REGRESSION MODELS COMPARISON:
----------------------------------------
                    Model  Train_R2  Test_R2   RMSE    MAE  CV_Mean
        Linear Regression    0.8048   0.8071 0.0510 0.0406   0.8020
Support Vector Regression    0.8015   0.8058 0.0511 0.0408   0.7992
                  XGBoost    0.8418   0.7796 0.0545 0.0432   0.7767

📊 CLASSIFICATION MODELS COMPARISON:
----------------------------------------
               Model  Train_Accuracy  Test_Accuracy  CV_Mean
 Logistic Regression          0.9107         0.9110   0.9027
             XGBoost          0.9890         0.9090   0.8965
Gaussian Naive Bayes          0.8935         0.8960   0.8882
       Random Forest          0.9948         0.8870   0.8747

 Best Regression Model: Linear Regression
 Best Classification Model: Logistic Regression


In [ ]:
def perform_unsupervised_learning(X_scaled):
    """Perform clustering and anomaly detection"""
    print("\n" + "="*60)
    print("UNSUPERVISED LEARNING")
    print("="*60)

    results = {}

    # K-Means Clustering
    print("\n K-Means Clustering...")
    kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
    cluster_labels = kmeans.fit_predict(X_scaled)

    silhouette = silhouette_score(X_scaled, cluster_labels)
    davies_bouldin = davies_bouldin_score(X_scaled, cluster_labels)

    print(f" Silhouette Score: {silhouette:.4f}")
    print(f" Davies-Bouldin Score: {davies_bouldin:.4f}")

    results['kmeans'] = {
        'model': kmeans,
        'labels': cluster_labels,
        'silhouette_score': silhouette,
        'davies_bouldin_score': davies_bouldin
    }

    # DBSCAN Clustering
    print("\n DBSCAN Clustering...")
    dbscan = DBSCAN(eps=0.5, min_samples=5)
    dbscan_labels = dbscan.fit_predict(X_scaled)

    n_clusters = len(set(dbscan_labels)) - (1 if -1 in dbscan_labels else 0)
    n_noise = list(dbscan_labels).count(-1)

    print(f" Number of clusters: {n_clusters}")
    print(f" Number of noise points: {n_noise}")

    results['dbscan'] = {
        'model': dbscan,
        'labels': dbscan_labels,
        'n_clusters': n_clusters,
        'n_noise': n_noise
    }

    # Anomaly Detection with Isolation Forest
    print("\n Anomaly Detection (Isolation Forest)...")
    iso_forest = IsolationForest(contamination=0.1, random_state=42)
    anomaly_labels = iso_forest.fit_predict(X_scaled)

    n_anomalies = list(anomaly_labels).count(-1)
    anomaly_rate = n_anomalies / len(anomaly_labels)

    print(f" Number of anomalies: {n_anomalies}")
    print(f" Anomaly rate: {anomaly_rate:.2%}")

    results['isolation_forest'] = {
        'model': iso_forest,
        'labels': anomaly_labels,
        'n_anomalies': n_anomalies,
        'anomaly_rate': anomaly_rate
    }

    return results

# Perform unsupervised learning
unsupervised_results = perform_unsupervised_learning(X_train_reg_scaled)


UNSUPERVISED LEARNING

 K-Means Clustering...
 Silhouette Score: 0.0374
 Davies-Bouldin Score: 4.2261

 DBSCAN Clustering...
 Number of clusters: 0
 Number of noise points: 4000

 Anomaly Detection (Isolation Forest)...
 Number of anomalies: 400
 Anomaly rate: 10.00%


In [ ]:
def explain_model_predictions(model, X_train, X_test, feature_names, model_name):
    """Use SHAP to explain model predictions"""
    print(f"\n" + "="*60)
    print(f"EXPLAINABLE AI - {model_name}")
    print("="*60)

    try:
        # Create SHAP explainer
        if hasattr(model, 'predict_proba'):
            explainer = shap.Explainer(model.predict_proba, X_train[:100])
        else:
            explainer = shap.Explainer(model.predict, X_train[:100])

        # Calculate SHAP values for test set sample
        shap_values = explainer(X_test[:50])

        print(" SHAP values calculated successfully")

        # Calculate feature importance
        if len(shap_values.shape) == 3:  # Multi-class classification
            mean_shap = np.abs(shap_values.values).mean(axis=(0, 2))
        else:  # Regression or binary classification
            mean_shap = np.abs(shap_values.values).mean(axis=0)

        # Create feature importance DataFrame
        feature_importance = pd.DataFrame({
            'feature': feature_names,
            'importance': mean_shap
        }).sort_values('importance', ascending=False)

        print("\n Top 10 Most Important Features:")
        print(feature_importance.head(10).to_string(index=False, float_format='%.4f'))

        return feature_importance, shap_values

    except Exception as e:
        print(f" SHAP analysis failed: {str(e)}")
        return None, None

# Explain best regression model
if best_reg_model in regression_models:
    reg_feature_importance, reg_shap_values = explain_model_predictions(
        regression_models[best_reg_model],
        X_train_reg_scaled,
        X_test_reg_scaled,
        list(X_reg.columns),
        best_reg_model
    )

# Explain best classification model
if best_clf_model in classification_models:
    clf_feature_importance, clf_shap_values = explain_model_predictions(
        classification_models[best_clf_model],
        X_train_clf_scaled,
        X_test_clf_scaled,
        list(X_clf.columns),
        best_clf_model
    )


EXPLAINABLE AI - Linear Regression
 SHAP values calculated successfully

 Top 10 Most Important Features:
              feature  importance
   Biodiversity_Index      0.2271
   Drought_Risk_Index      0.1235
     Flood_Risk_Index      0.1226
   Climate_Risk_Score      0.0909
    Soil_Erosion_Risk      0.0849
        Temperature_C      0.0022
Protected_Area_Status      0.0020
 Human_Activity_Index      0.0017
    Air_Quality_Index      0.0007
              Country      0.0003

EXPLAINABLE AI - Logistic Regression
 SHAP values calculated successfully

 Top 10 Most Important Features:
              feature  importance
   Biodiversity_Index      0.1906
   Drought_Risk_Index      0.0988
     Flood_Risk_Index      0.0972
   Climate_Risk_Score      0.0714
    Soil_Erosion_Risk      0.0660
 Human_Activity_Index      0.0070
  Accessibility_Score      0.0032
        Temperature_C      0.0019
Protected_Area_Status      0.0017
              Country      0.0017


In [ ]:
def predict_vulnerability(new_data, model, scaler, encoders, feature_columns, task_type='regression'):
    """
    Make predictions on new eco-tourism site data

    Parameters:
    - new_data: dictionary or DataFrame with site characteristics
    - model: trained model
    - scaler: fitted StandardScaler
    - encoders: dictionary of fitted LabelEncoders
    - feature_columns: list of feature column names
    - task_type: 'regression' or 'classification'
    """

    # Convert to DataFrame if dictionary
    if isinstance(new_data, dict):
        new_df = pd.DataFrame([new_data])
    else:
        new_df = new_data.copy()

    # Apply same preprocessing as training data
    processed_df = new_df.copy()

    # Encode categorical variables
    for col, encoder in encoders.items():
        if col in processed_df.columns:
            # Handle new categories not seen during training
            processed_df[col] = processed_df[col].astype(str)
            processed_df[col] = processed_df[col].apply(
                lambda x: encoder.transform([x])[0] if x in encoder.classes_ else 0
            )

    # Handle boolean columns
    if 'Protected_Area_Status' in processed_df.columns:
        processed_df['Protected_Area_Status'] = processed_df['Protected_Area_Status'].astype(int)

    # Select only feature columns
    feature_data = processed_df[feature_columns]

    # Scale features
    feature_data_scaled = scaler.transform(feature_data)

    # Make prediction
    if task_type == 'regression':
        prediction = model.predict(feature_data_scaled)
        return prediction[0] if len(prediction) == 1 else prediction
    else:
        prediction = model.predict(feature_data_scaled)
        prediction_proba = model.predict_proba(feature_data_scaled)

        risk_categories = ['Low', 'Medium', 'High']
        result = {
            'predicted_category': risk_categories[prediction[0]],
            'probabilities': {
                risk_categories[i]: prediction_proba[0][i]
                for i in range(len(risk_categories))
            }
        }
        return result

# Example usage function
def demonstrate_prediction():
    """Demonstrate how to use the prediction function"""
    print("PREDICTION DEMONSTRATION")

    # Sample a random row from the original DataFrame
    random_site_data = df.sample(n=1, random_state=np.random.randint(0, 10000)).iloc[0]

    # Example new site data
    new_site_random = {
        'Latitude': random_site_data['Latitude'],
        'Longitude': random_site_data['Longitude'],
        'Country': random_site_data['Country'],
        'Biodiversity_Index': random_site_data['Biodiversity_Index'],
        'Air_Quality_Index': random_site_data['Air_Quality_Index'],
        'Elevation_m': random_site_data['Elevation_m'],
        'Vegetation_Type': random_site_data['Vegetation_Type'],
        'Flood_Risk_Index': random_site_data['Flood_Risk_Index'],
        'Drought_Risk_Index': random_site_data['Drought_Risk_Index'],
        'Temperature_C': random_site_data['Temperature_C'],
        'Annual_Rainfall_mm': random_site_data['Annual_Rainfall_mm'],
        'Soil_Type': random_site_data['Soil_Type'],
        'Soil_Erosion_Risk': random_site_data['Soil_Erosion_Risk'],
        'Tourist_Capacity_Limit': random_site_data['Tourist_Capacity_Limit'],
        'Current_Tourist_Count': random_site_data['Current_Tourist_Count'],
        'Accessibility_Score': random_site_data['Accessibility_Score'],
        'Human_Activity_Index': random_site_data['Human_Activity_Index'],
        'Protected_Area_Status': random_site_data['Protected_Area_Status'],
        'Conservation_Investment_USD': random_site_data['Conservation_Investment_USD'],
        'Climate_Risk_Score': random_site_data['Climate_Risk_Score']
    }
    for key, value in new_site_random.items():
        print(f"'{key}': {value},")

   # Make regression prediction with the random site data
    if best_reg_model in regression_models:
        vulnerability_score_random = predict_vulnerability(
            new_site_random,
            regression_models[best_reg_model],
            scaler_reg,
            encoders_reg,
            list(X_reg.columns),
            task_type='regression'
        )
        print(f"\n📊 Predicted Vulnerability Score for random site: {vulnerability_score_random:.4f}")

    # Make classification prediction with the random site data
    if best_clf_model in classification_models:
        risk_prediction_random = predict_vulnerability(
            new_site_random,
            classification_models[best_clf_model],
            scaler_clf,
            encoders_clf,
            list(X_clf.columns),
            task_type='classification'
        )
        print(f"\n Predicted Risk Category for random site: {risk_prediction_random['predicted_category']}")
        print("📊 Risk Probabilities for random site:")
        for category, prob in risk_prediction_random['probabilities'].items():
            print(f"   {category}: {prob:.4f}")
demonstrate_prediction()

PREDICTION DEMONSTRATION
'Latitude': 19.846504319710057,
'Longitude': -79.38199091943099,
'Country': USA,
'Biodiversity_Index': 0.6300489337797761,
'Air_Quality_Index': 187.62578830201724,
'Elevation_m': 1374.9987375008575,
'Vegetation_Type': Wetland,
'Flood_Risk_Index': 0.7183069185922475,
'Drought_Risk_Index': 0.03148983286726992,
'Temperature_C': 27.0910967372344,
'Annual_Rainfall_mm': 559.3513196513468,
'Soil_Type': Clay,
'Soil_Erosion_Risk': 0.5525539988934401,
'Tourist_Capacity_Limit': 4339.187799266516,
'Current_Tourist_Count': 2696.4698641548002,
'Accessibility_Score': 0.925402927389734,
'Human_Activity_Index': 0.18740610611869873,
'Protected_Area_Status': True,
'Conservation_Investment_USD': 63231.3411502071,
'Climate_Risk_Score': 0.7840969311016962,

📊 Predicted Vulnerability Score for random site: 0.2310

 Predicted Risk Category for random site: Medium
📊 Risk Probabilities for random site:
   Low: 0.0000
   Medium: 0.9769
   High: 0.0231


In [ ]:
import joblib
import json

def save_models_and_results():
    """Save trained models and results for future use"""
    print("\\n" + "="*60)
    print("SAVING MODELS AND RESULTS")
    print("="*60)

    # Save models
    if best_reg_model in regression_models:
        joblib.dump(regression_models[best_reg_model], f'best_regression_model_{best_reg_model.lower().replace(" ", "_")}.pkl')
        joblib.dump(scaler_reg, 'regression_scaler.pkl')
        print(f" Saved regression model: {best_reg_model}")

    if best_clf_model in classification_models:
        joblib.dump(classification_models[best_clf_model], f'best_classification_model_{best_clf_model.lower().replace(" ", "_")}.pkl')
        joblib.dump(scaler_clf, 'classification_scaler.pkl')
        print(f" Saved classification model: {best_clf_model}")

    # Save encoders
    joblib.dump(encoders_reg, 'regression_encoders.pkl')
    joblib.dump(encoders_clf, 'classification_encoders.pkl')
    print(" Saved label encoders")

    # Save feature names
    with open('feature_names.json', 'w') as f:
        json.dump({
            'regression_features': list(X_reg.columns),
            'classification_features': list(X_clf.columns)
        }, f)
    print(" Saved feature names")

    # Save results summary
    results_summary = {
        'regression_results': {k: {key: float(val) if isinstance(val, (int, float, np.number)) else str(val)
                                  for key, val in v.items() if key != 'predictions'}
                             for k, v in regression_results.items()},
        'classification_results': {k: {key: float(val) if isinstance(val, (int, float, np.number)) else str(val)
                                     for key, val in v.items() if key not in ['predictions', 'classification_report']}
                                 for k, v in classification_results.items()},
        'best_models': {
            'regression': best_reg_model,
            'classification': best_clf_model
        }
    }

    with open('model_results_summary.json', 'w') as f:
        json.dump(results_summary, f, indent=2)
    print(" Saved results summary")

In [ ]:
import joblib
import json

def save_models_and_results():
    """Save trained models and results for future use"""
    print("\n" + "="*60)
    print("SAVING MODELS AND RESULTS")
    print("="*60)

    # Save models
    if best_reg_model in regression_models:
        joblib.dump(regression_models[best_reg_model], f'best_regression_model_{best_reg_model.lower().replace(" ", "_")}.pkl')
        joblib.dump(scaler_reg, 'regression_scaler.pkl')
        print(f" Saved regression model: {best_reg_model}")

    if best_clf_model in classification_models:
        joblib.dump(classification_models[best_clf_model], f'best_classification_model_{best_clf_model.lower().replace(" ", "_")}.pkl')
        joblib.dump(scaler_clf, 'classification_scaler.pkl')
        print(f" Saved classification model: {best_clf_model}")

    # Save encoders
    joblib.dump(encoders_reg, 'regression_encoders.pkl')
    joblib.dump(encoders_clf, 'classification_encoders.pkl')
    print(" Saved label encoders")

    # Save feature names
    with open('feature_names.json', 'w') as f:
        json.dump({
            'regression_features': list(X_reg.columns),
            'classification_features': list(X_clf.columns)
        }, f)
    print(" Saved feature names")

    # Save results summary
    results_summary = {
        'regression_results': {k: {key: float(val) if isinstance(val, (int, float, np.number)) else str(val)
                                  for key, val in v.items() if key != 'predictions'}
                             for k, v in regression_results.items()},
        'classification_results': {k: {key: float(val) if isinstance(val, (int, float, np.number)) else str(val)
                                     for key, val in v.items() if key not in ['predictions', 'classification_report']}
                                 for k, v in classification_results.items()},
        'best_models': {
            'regression': best_reg_model,
            'classification': best_clf_model
        }
    }

    with open('model_results_summary.json', 'w') as f:
        json.dump(results_summary, f, indent=2)
    print(" Saved results summary")

# Save everything
save_models_and_results()


SAVING MODELS AND RESULTS
 Saved regression model: Linear Regression
 Saved classification model: Logistic Regression
 Saved label encoders
 Saved feature names
 Saved results summary
